In [10]:
import pandas as pd
import numpy as np

## 1. Load Data & Select Columns

In [11]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

cols = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]
df = df[cols].copy()


## EDA Questions

In [12]:
print("--- Question 1 ---")
print("Missing values per column:")
print(df.isnull().sum())  

print("\n--- Question 2 ---")
print(f"Horsepower Median: {df['horsepower'].median()}") 

--- Question 1 ---
Missing values per column:
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

--- Question 2 ---
Horsepower Median: 254.0


## Helper Functions

In [13]:
def prepare_X(df_subset, features, fill_value):
    df_num = df_subset[features].copy()
    df_num = df_num.fillna(fill_value)
    return df_num.values

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

## Feature list & target setup

In [14]:
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

In [15]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

## Question 3

In [16]:
print("\n--- Question 3 ---")
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)


--- Question 3 ---


In [17]:
df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()

In [18]:
y_train = df_train[target].values
y_val = df_val[target].values

In [19]:
X_train_0 = prepare_X(df_train, features, fill_value=0)
w0_0, w_0 = train_linear_regression(X_train_0, y_train)

In [20]:
X_val_0 = prepare_X(df_val, features, fill_value=0)
y_pred_0 = w0_0 + X_val_0.dot(w_0)
score_0 = round(rmse(y_val, y_pred_0), 3)

In [21]:
mean_hp = df_train['horsepower'].mean()
X_train_mean = prepare_X(df_train, features, fill_value=mean_hp)
w0_m, w_m = train_linear_regression(X_train_mean, y_train)

In [22]:
X_val_mean = prepare_X(df_val, features, fill_value=mean_hp)
y_pred_m = w0_m + X_val_mean.dot(w_m)
score_m = round(rmse(y_val, y_pred_m), 3)

In [23]:
print(f"RMSE (fill with 0): {score_0}")
print(f"RMSE (fill with mean): {score_m}")

RMSE (fill with 0): 2.205
RMSE (fill with mean): 2.202


## Question 4: Regularization

In [24]:
print("\n--- Question 4 ---")
X_train_reg = prepare_X(df_train, features, fill_value=0)
X_val_reg = prepare_X(df_val, features, fill_value=0)


--- Question 4 ---


In [25]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
for r in r_values:
    w0, w = train_linear_regression_reg(X_train_reg, y_train, r=r)
    y_pred = w0 + X_val_reg.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:<5}: RMSE = {score}")

r = 0    : RMSE = 2.2053
r = 0.01 : RMSE = 2.2058
r = 0.1  : RMSE = 2.2241
r = 1    : RMSE = 2.3492
r = 5    : RMSE = 2.4094
r = 10   : RMSE = 2.4195
r = 100  : RMSE = 2.4292


## Question 5: Seed Sensitivity

In [26]:
print("\n--- Question 5 ---")
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_list = []



--- Question 5 ---


In [27]:
for s in seeds:
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_tr = df.iloc[idx[:n_train]]
    df_v = df.iloc[idx[n_train:n_train + n_val]]

    y_tr = df_tr[target].values
    y_v = df_v[target].values

    X_tr = prepare_X(df_tr, features, fill_value=0)
    X_v = prepare_X(df_v, features, fill_value=0)

    w0, w = train_linear_regression(X_tr, y_tr)
    y_pred = w0 + X_v.dot(w)
    rmse_list.append(rmse(y_v, y_pred))

In [28]:
std_score = round(np.std(rmse_list), 3)
print(f"Standard Deviation of RMSE scores across seeds: {std_score}")

Standard Deviation of RMSE scores across seeds: 0.029


## Question 6: Test Dataset Evaluation (Seed 9)

In [29]:
print("\n--- Question 6 ---")
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)


--- Question 6 ---


In [30]:
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [31]:
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

In [32]:
y_full_train = df_full_train[target].values
y_test = df_test[target].values

In [33]:
X_full_train = prepare_X(df_full_train, features, fill_value=0)
X_test = prepare_X(df_test, features, fill_value=0)

In [34]:
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)
test_rmse = round(rmse(y_test, y_pred), 3)

In [35]:
print(f"Test RMSE (r=0.001, seed=9): {test_rmse}")

Test RMSE (r=0.001, seed=9): 2.236
